# Optiver Realized Volatility — Exploration

In [1]:
import pandas as pd

train = pd.read_csv("data/train.csv")
print(train.shape)
print(train.columns)
train.head()

(428932, 3)
Index(['stock_id', 'time_id', 'target'], dtype='str')


,stock_id,time_id,target
0,0,5,0.004136
1,0,11,0.001445
2,0,16,0.002168
3,0,31,0.002195
4,0,62,0.001747


In [2]:
# book_train.parquet is partitioned by stock_id; only loading stock 0 for now.
book_stock_0 = pd.read_parquet("data/book_train.parquet/stock_id=0")
print(book_stock_0.shape)
print(book_stock_0.columns)
book_stock_0.head()

(917553, 10)
Index(['time_id', 'seconds_in_bucket', 'bid_price1', 'ask_price1',
       'bid_price2', 'ask_price2', 'bid_size1', 'ask_size1', 'bid_size2',
       'ask_size2'],
      dtype='str')


,time_id,seconds_in_bucket,bid_price1,ask_price1,bid_price2,ask_price2,bid_size1,ask_size1,bid_size2,ask_size2
0,5,0,1.001422,1.002301,1.00137,1.002353,3,226,2,100
1,5,1,1.001422,1.002301,1.00137,1.002353,3,100,2,100
2,5,5,1.001422,1.002301,1.00137,1.002405,3,100,2,100
3,5,6,1.001422,1.002301,1.00137,1.002405,3,126,2,100
4,5,7,1.001422,1.002301,1.00137,1.002405,3,126,2,100


In [3]:
one_window=book_stock_0[book_stock_0["time_id"]==5]
print(one_window.head())

   time_id  seconds_in_bucket  bid_price1  ask_price1  bid_price2  ask_price2  \
0        5                  0    1.001422    1.002301     1.00137    1.002353   
1        5                  1    1.001422    1.002301     1.00137    1.002353   
2        5                  5    1.001422    1.002301     1.00137    1.002405   
3        5                  6    1.001422    1.002301     1.00137    1.002405   
4        5                  7    1.001422    1.002301     1.00137    1.002405   

   bid_size1  ask_size1  bid_size2  ask_size2  
0          3        226          2        100  
1          3        100          2        100  
2          3        100          2        100  
3          3        126          2        100  
4          3        126          2        100  


In [4]:
# numerator: cross-weight each side's price by the OTHER side's size, then add
one_window["wap"]=(
     (one_window["bid_price1"] * one_window["ask_size1"]  # best buy price weighted by sell-side size
     + one_window["ask_price1"] * one_window["bid_size1"])  # + best sell price weighted by buy-side size
    / (one_window["bid_size1"] + one_window["ask_size1"])  # divide by total size at the best bid+ask
)
import numpy as np
#return is how much the price moved from one moment to the next 
#we use log because it is additive and symmetric so a move up then an equal move down will cancel to 0
one_window["log_return"]=np.log(one_window["wap"]/one_window["wap"].shift(1))
print(one_window[["seconds_in_bucket", "wap", "log_return"]].head())

   seconds_in_bucket       wap  log_return
0                  0  1.001434         NaN
1                  1  1.001448    0.000014
2                  5  1.001448    0.000000
3                  6  1.001443   -0.000005
4                  7  1.001443    0.000000


In [5]:
#realized volatility
realized_vol=np.sqrt=((one_window["log_return"]**2).sum())
print("realized vol (computed):", realized_vol)
#get row with both stock_id=0 and time_id=5 for target
target_row=train[(train["stock_id"]==0) & (train["time_id"]==5)]
target_value=target_row["target"].iloc[0]
print("target from train.csv: ", target_value )

realized vol (computed): 2.0244277959355353e-05
target from train.csv:  0.004135767
